# 02 · Проверка модели перед экспериментами

Один раз на каждую новую модель или новый chat template. Ничего не пишет в журнал,
кроме заметки в конце (по желанию).

Проверяем:
1. какой Python и устройство (MPS на Mac);
2. **окна позиций глазами** — что `field_read`, `post_message`, `pre_action`
   стоят там, где должны (см. AGENTS.md: на Qwen3.x арифметика окон уже ломалась);
3. формат вызова инструмента, выведенный из шаблона;
4. что хук steering меняет ровно столько позиций, сколько в окне;
5. сколько времени займёт полный прогон.

In [ ]:
%load_ext autoreload
%autoreload 2
import os
os.environ.setdefault("PYTORCH_ENABLE_MPS_FALLBACK", "1")   # Mac: неподдержанные операции -> CPU

import pandas as pd
from IPython.display import Markdown, display
pd.set_option("display.precision", 3)
pd.set_option("display.width", 200)

from steering_tool_feedback.config import ExperimentConfig
from steering_tool_feedback.pipeline import Experiment
from steering_tool_feedback import journal, viz
from steering_tool_feedback import doctor, simulator as S, steering as ST
from steering_tool_feedback.windows import describe_windows
import numpy as np
doctor.main([])

In [ ]:
MODEL = "Qwen/Qwen3-1.7B"     # HF id или локальная папка; меняй здесь
cfg = ExperimentConfig.make(name="model-check", hypothesis="explore",
                            notebook="notebooks/02_model_check.ipynb",
                            overrides={"model.name": MODEL})
exp = Experiment(cfg)
exp.data()
be = exp.backend
be.info()

### Окна позиций — смотреть глазами

In [ ]:
t = next(x for x in exp.state["splits"]["train"] if x.pair_type == "changing")
for b in (0, 1):
    p = exp._prompt(t, b)
    print(f"--- {t.task_id} branch {b}: allowed {sorted(t.branch(b).allowed_actions)}")
    if p.prepared is not None:
        print(describe_windows(be.tok, p.prepared))
print("action format:", be.info()["action_format"])

### Хук срабатывает ровно в окне на этой модели

In [ ]:
from steering_tool_feedback.backends import ResolvedSteer
p = exp._prompt(t, 1)
vec = ST.SteeringVector(be.n_layers // 2, np.ones(be.hidden_size, dtype=np.float32), "post_message", {})
for kind in ("field_read", "post_message", "pre_action"):
    w = p.windows[kind]
    scores, hook = be.score_actions(p.prepared.input_ids, ResolvedSteer(vec, w, 1.0))
    print(f"{kind:13} window={len(w):3} applied={hook.n_applied:3} positions from {hook.positions_source}")

### Что модель выбирает без вмешательства и сколько это стоит по времени

In [ ]:
for b in (0, 1):
    d = be.decide(exp._prompt(t, b), "scored", None, 0.0, 0)
    top = sorted(d.scores.items(), key=lambda kv: -kv[1])[:3]
    print(f"branch {b}: {d.action:20} ok={d.action in t.branch(b).allowed_actions}  "
          f"{d.latency_ms:.0f} ms  top3={[(a, round(s, 2)) for a, s in top]}")
exp.estimate()

In [ ]:
# По желанию — зафиксировать в журнале, что модель проверена:
# journal.note(f"Модель {MODEL} проверена: окна ок, формат {be.info()['action_format']}, "
#              f"оценка полного прогона ~N мин на {be.info()['device']}.", tag="модель")